# Complexity, Floating Point & Convergence: Practice Notebook

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

## How to work through the problems

1. Read the goal and the **pseudocode** (a plain-language recipe).
2. Write the Python yourself in the empty cell below it.
3. Run the **mini-check**. If it prints `OK` (or `passed`), go on.

Stuck? Open a **Hint**. Try first!

| pseudocode | Python |
|---|---|
| `x ← 0` | `x = 0` |
| `for each a in L` | `for a in L:` |
| `repeat N times` | `for k in range(N):` |
| `while condition` | `while condition:` |
| `append v to L` | `L.append(v)` |
| `L[k]`, `last entry of L` | `L[k]`, `L[-1]` |
| `abs(v)`, `log(v)`, `cos(v)`, `sqrt(v)` | `abs(v)`, `np.log(v)`, `np.cos(v)`, `np.sqrt(v)` |
| `10^k` | `10.0**k` |
| `make an array from list L` | `np.array(L)` |
| `floor(v)`, `log10(v)`, `round(v)` | `np.floor(v)`, `np.log10(v)`, `np.round(v)` |
| `as an integer` | `int(...)` |

---
## Warm-up: scientific notation

Every number can be written as

$$x = \pm\, m \times 10^{\,e}, \qquad 1 \le m < 10$$

with the **mantissa** $m$ (the digits) and the **exponent** $e$ (the size). Python writes $4.2 \times 10^{-4}$ as `4.2e-4`.
A float is exactly this idea, in base 2, with a fixed number of digits.

### Step 1: read and run

Nothing to write. Look at how Python prints and reads scientific notation.

In [ ]:
print(2.99792458e8)                 # the speed of light in m/s
print(4.2e-4 == 0.00042)            # the same number, written two ways
print(f"{299792458:.3e}")           # print with 3 digits after the point
print(f"{0.000123456:.2e}")


### Step 2: `sci(x)` splits a number into mantissa and exponent

The exponent is "how many times 10 fits in", which is the integer part of $\log_{10}|x|$.

```
function sci(x):
    e ← floor(log10(abs(x))), as an integer
    m ← x / 10^e
    return [m, e]
```


<details><summary><b>Hint</b></summary>

For $x = 0.00042$: $\log_{10}(0.00042) \approx -3.38$, and the floor is $-4$. Then $m = 0.00042 / 10^{-4} = 4.2$.

</details>

In [ ]:
def sci(x):
    # your code here
    pass

In [ ]:
# mini-check
for x, m_true, e_true in [[6.02e23, 6.02, 23], [0.00042, 4.2, -4], [-1500.0, -1.5, 3]]:
    m, e = sci(x)
    assert e == e_true and abs(m - m_true) < 1e-12, [x, m, e]
    print(f"{x} = {m} x 10^{e}")
print("OK")

### Step 3: `round_sig(x, d)` keeps only $d$ significant digits

This is what a computer with $d$ digits has to do with every number.

```
function round_sig(x, d):
    e ← floor(log10(abs(x)))
    scale ← 10^(e - d + 1)
    return round(x / scale) * scale
```


<details><summary><b>Hint</b></summary>

For $x = 123456$ and $d = 2$: $e = 5$, scale $= 10^{4}$, $x / \text{scale} = 12.3456$ rounds to $12$, and $12 \times 10^4 = 120000$.

</details>

In [ ]:
def round_sig(x, d):
    # your code here
    pass

In [ ]:
# mini-check
assert abs(round_sig(np.pi, 3) - 3.14) < 1e-12
assert round_sig(123456.0, 2) == 120000.0
assert abs(round_sig(1 / 3, 3) - 0.333) < 1e-12
print("OK")

### Step 4: the error of a toy 3-digit computer

How big can the **relative** rounding error be when every number keeps only 3 digits?

```
xs ← 10000 random numbers between 1 and 1000     (np.random.default_rng(seed=47).uniform(1, 1000, 10000))
rel_err ← for each x in xs: abs(round_sig(x, 3) - x) / x
print the largest value in rel_err
```


<details><summary><b>Hint</b></summary>

`[abs(round_sig(x, 3) - x) / x for x in xs]` builds the list; `max(...)` gives the largest value.

</details>

In [ ]:
# your code here

---
## Demo 1: floating point surprises

Nothing to write: run and read. A Python `float` uses 64 bits, so only finitely many numbers can be stored and everything else is **rounded**.

In [ ]:
print("0.1 + 0.2        =", 0.1 + 0.2)
print("0.1 + 0.2 == 0.3 :", 0.1 + 0.2 == 0.3)
print("tolerance test   :", abs((0.1 + 0.2) - 0.3) < 1e-12)
print()
print("machine epsilon for float32 =", np.finfo(np.float32).eps)  # machine epsilon for float32
print("machine epsilon for float64 =", np.finfo(np.float64).eps)  # machine epsilon for float64
print()
print("machine epsilon  =", np.finfo(float).eps)  # machine epsilon for the default float type
print("largest float    =", np.finfo(float).max)  # largest representable float
print("smallest normal  =", np.finfo(float).tiny)  # smallest normal float
print()
print("1e16 + 1 - 1e16  =", 1e16 + 1 - 1e16)
print("1e308 * 10       =", 1e308 * 10)
print("inf - inf        =", np.inf - np.inf)
print("nan == nan       :", np.nan == np.nan)

**Questions:** why is `1e16 + 1 - 1e16` equal to `0.0`? Why does a test with a tolerance work when `==` fails?


<details><summary><b>Check your answer</b></summary>

Near $10^{16}$ the gap between floats is $2$, so $10^{16} + 1$ rounds back to $10^{16}$.
$0.1 + 0.2$ is off by only about $4 \times 10^{-17}$: not *equal* to $0.3$, but well within any sensible tolerance.

</details>

---
## Demo 2: $(1 + 1/n)^n \to e$ ... until floating point gets in the way

In exact arithmetic the error $|(1+1/n)^n - e|$ is $O(1/n)$: ten times bigger $n$, ten times smaller error. Watch what the computer does.

In [ ]:
ns = 10.0 ** np.arange(1, 17)
values = (1 + 1 / ns) ** ns
errors = np.abs(values - np.e)

for n, v, e in zip(ns, values, errors):
    print(f"n = {n:8.0e}   (1 + 1/n)^n = {v:.12f}   error = {e:.1e}")

plt.loglog(ns, errors, "o-", label="computed error")
plt.loglog(ns, np.e / (2 * ns), "--", label="e / (2n)  (what theory predicts)")
plt.xlabel("n"); plt.ylabel("error"); plt.legend(); plt.show()

Up to about $n = 10^8$ the error follows $O(1/n)$. After that it **grows**: $1 + 1/n$ is rounded, and raising it to a huge power magnifies that rounding error. At $n = 10^{16}$, $1 + 10^{-16}$ rounds to exactly $1$ and the answer is $1$.

---
## Problem 1: Big-$O$ by doubling

**Goal:** find the growth of a loop without any theory: count its **flops** (floating point operations: $+, -, \times, \div$), double $n$, and see how the flop count changes.

| when $n$ doubles, the flop count ... | Big-$O$ |
|---|---|
| grows by about 1 | $O(\log n)$ |
| doubles (×2) | $O(n)$ |
| ×4 | $O(n^2)$ |
| ×8 | $O(n^3)$ |

In [ ]:
# Each function does some floating point work (total = total + 1.0)
# and returns how many flops it performed.

def flops_a(n):
    total = 0.0
    flops = 0
    for i in range(n):
        for j in range(n):
            total = total + 1.0     # 1 flop
            flops += 1
    return flops

def flops_b(n):
    total = 0.0
    flops = 0
    i = 1
    while i < n:
        total = total + 1.0         # 1 flop
        flops += 1
        i *= 2
    return flops

def flops_c(n):
    total = 0.0
    flops = 0
    for i in range(n):
        total = total + 1.0         # 1 flop
        flops += 1
    for i in range(n):
        total = total * 1.0         # 1 flop
        flops += 1
    return flops

### Step 1: guess (no code)

Read `flops_a`, `flops_b` and `flops_c`. How many flops does each do, and what Big-$O$ do you expect?

### Step 2: `doubling_factor(flops, n)`

`flops` is one of the functions above; the result says how much the flop count grows when $n$ doubles.

```
function doubling_factor(flops, n):
    return flops(2 * n) / flops(n)
```

In [ ]:
def doubling_factor(flops, n):
    # your code here
    pass

In [ ]:
# mini-check
assert doubling_factor(flops_a, 100) == 4.0
assert doubling_factor(flops_c, 100) == 2.0
print("OK")

### Step 3: use it

```
for each flops in [flops_a, flops_b, flops_c]:
    print the name of flops, and doubling_factor(flops, 1000)
```

Compare with the table above and with your guesses.


<details><summary><b>Hint</b></summary>

`flops.__name__` gives the name of a function as text.

</details>

In [ ]:
# your code here

<details><summary><b>Check your answer</b></summary>

`flops_a`: $n^2$ flops, ×4, so $O(n^2)$ (two nested loops). `flops_b`: the flop count goes from 10 to 11, so $O(\log n)$.
`flops_c`: $2n$ flops, ×2, so $O(n)$: two loops **one after the other** add up, and the constant 2 is dropped.

</details>

---
## Problem 2: machine epsilon and the gaps between floats

**Goal:** find the smallest number $\varepsilon$ with $1 + \varepsilon > 1$ on your computer, then look at the gaps between floats elsewhere on the number line.

### Step 1: `machine_epsilon()`

Keep halving `eps` as long as adding **half of it** to 1 still changes the result.


```
function machine_epsilon():
    eps ← 1.0
    ncount ← 0.0
    while 1.0 + eps / 2 > 1.0:
        eps ← eps / 2
        ncount ← ncount +1
    print ncount as the number of iterations to find machine epsilon
    return eps
```

In [ ]:
def machine_epsilon():
    # your code here
    pass

In [ ]:
# mini-check
eps = machine_epsilon()
print("your epsilon :", eps)
print("numpy's epsilon:", np.finfo(float).eps)
assert eps == np.finfo(float).eps == 2.0**-52
print("OK")

**Question:** how many times did the loop halve `eps`? What does that number have to do with the 52 fraction bits?


<details><summary><b>Check your answer</b></summary>

$\varepsilon = 2^{-52}$, so the loop halved 52 times: one halving per fraction bit.

</details>

### Step 2: the gap grows with the number

`np.spacing(x)` is the gap between `x` and the next float. Print a small table:

```
for each x in [1.0, 1000.0, 1e8, 1e16, 1e300]:
    gap ← np.spacing(x)
    print x, gap, and the relative gap gap / x
```


<details><summary><b>Hint</b></summary>

`f"{x:.0e}"` and `f"{gap:.2e}"` print numbers in short scientific notation.

</details>

In [ ]:
# your code here

**Question:** the gap changes enormously. What about `gap / x`?


<details><summary><b>Check your answer</b></summary>

The relative gap always stays between $\varepsilon/2$ and $\varepsilon$ (about $1.1 \times 10^{-16}$ to $2.2 \times 10^{-16}$).
That is why we say a float has about **16 significant digits**, whatever its size.

</details>

### Step 3: the order of addition matters

Add $1.0$ ten thousand times to $10^{16}$, in two different orders:

```
function big_first():
    total ← 1e16
    repeat 10000 times:
        total ← total + 1.0
    return total

function small_first():
    total ← 0.0
    repeat 10000 times:
        total ← total + 1.0
    return total + 1e16
```

In [ ]:
def big_first():
    # your code here
    pass


def small_first():
    # your code here
    pass

In [ ]:
# mini-check
print("big first   :", big_first() - 1e16)
print("small first :", small_first() - 1e16)
assert big_first() == 1e16
assert small_first() == 1e16 + 10000
print("Problem 2: passed")

<details><summary><b>Why?</b></summary>

With the big number first, every single `+ 1.0` is smaller than half the gap at $10^{16}$ and is rounded away: all 10000 ones are lost.
Adding the small numbers first lets them grow to $10^4$, which is big enough to survive. Rule: **add small numbers first**.

</details>

---
## Problem 3: catastrophic cancellation

**Goal:** see how subtracting two nearly equal numbers destroys accuracy, and fix it by rewriting the formula.

$$f(x) = \frac{1 - \cos x}{x^2} \;\to\; \frac12 \quad (x \to 0), \qquad \text{same function: } f(x) = \frac{2\sin^2(x/2)}{x^2}$$

### Step 1: two formulas for the same function

```
function f_naive(x):
    return (1 - cos(x)) / x^2

function f_stable(x):
    return 2 * sin(x / 2)^2 / x^2
```


<details><summary><b>Hint</b></summary>

Use `np.cos` and `np.sin`, so the functions also work on whole arrays. `x^2` is `x**2`.

</details>

In [ ]:
def f_naive(x):
    # your code here
    pass


def f_stable(x):
    # your code here
    pass

In [ ]:
# mini-check
assert abs(f_naive(0.5) - f_stable(0.5)) < 1e-14      # for moderate x they agree
assert f_naive(1e-8) == 0.0                            # for tiny x the naive one collapses
assert abs(f_stable(1e-8) - 0.5) < 1e-15              # the stable one gives the correct answer
print("OK")

### Step 2: measure the damage

```
xs ← the array 10^-1, 10^-2, ..., 10^-8        (np.logspace(-1, -8, 8))
rel_err ← abs(f_naive(xs) - f_stable(xs)) / f_stable(xs)
print xs and rel_err side by side
plot rel_err against xs on log-log axes
```


<details><summary><b>Hint</b></summary>

Because `f_naive` and `f_stable` use NumPy, you can pass the whole array `xs` at once: no loop needed for `rel_err`.
To print side by side, loop with `for x, r in zip(xs, rel_err):`.

To plot a log-log axes, use 
```
fig, ax = plt.subplots()
ax.loglog(x,y)
```
 and to invert your x-axis, use `ax.invert_xaxis()`

</details>

In [ ]:
# your code here

**Question:** roughly how many correct digits does the naive formula have at $x = 10^{-4}$? And at $x = 10^{-8}$?


<details><summary><b>Check your answer</b></summary>

At $x = 10^{-4}$ the relative error is about $10^{-8}$: only about 8 of the 16 digits are left. At $10^{-8}$ it is $100\%$: none.
Why: $\cos(10^{-4}) = 0.999999995\ldots$, so $1 - \cos x$ throws away the 8 leading digits, which were all 9's.

</details>

### Step 3: your turn to fix a formula

$g(x) = \sqrt{x + 1} - \sqrt{x}$ is small for large $x$ and suffers from cancellation.
Multiplying top and bottom by $\sqrt{x+1} + \sqrt{x}$ gives the same function without subtraction:

$$g(x) = \frac{1}{\sqrt{x + 1} + \sqrt{x}}$$

```
function g_naive(x):
    return sqrt(x + 1) - sqrt(x)

function g_stable(x):
    return 1 / (sqrt(x + 1) + sqrt(x))

for each x in [1e2, 1e6, 1e10, 1e14]:
    print x, g_naive(x), g_stable(x), and the relative error of g_naive
```
with relative error of $g_n$ and $g_s$ being $\frac{|g_n-g_s|}{g_s}$. 

In [ ]:
def g_naive(x):
    # your code here
    pass


def g_stable(x):
    # your code here
    pass


# your code here

In [ ]:
# mini-check
assert abs(g_stable(1e14) - 0.5e-7) / 0.5e-7 < 1e-6       # close to 1 / (2 sqrt(x))
assert abs(g_naive(1e14) - g_stable(1e14)) / g_stable(1e14) > 1e-4
print("Problem 3: passed")

---
## Problem 4: how fast does a sequence converge?

**Goal:** generate two sequences, measure their errors $e_n = |x_n - L|$, and find out **how fast** they converge.

- **Fixed point iteration:** $x_{n+1} = \cos(x_n)$, which converges to $L = 0.7390851332151607\ldots$
- **Newton's method for $\sqrt 2$:** $x_{n+1} = \frac12\left(x_n + \frac{2}{x_n}\right)$, which converges to $L = \sqrt 2$.

Both are of the form $x_{n+1} = g(x_n)$:

In [ ]:
def g_cos(x):
    return np.cos(x)

def g_newton(x):
    return (x + 2 / x) / 2

L_cos = 0.7390851332151607
L_newton = np.sqrt(2)

### Step 1: `iterate(g, x0, N)`

Return the list $[x_0, x_1, \ldots, x_N]$.

```
function iterate(g, x0, N):
    xs ← [x0]
    repeat N times:
        append g(last entry of xs) to xs
    return xs
```

In [ ]:
def iterate(g, x0, N):
    # your code here
    pass

In [ ]:
# mini-check
assert iterate(g_newton, 1.0, 2) == [1.0, 1.5, (1.5 + 2 / 1.5) / 2]
assert len(iterate(g_cos, 1.0, 10)) == 11
print("OK")

### Step 2: errors and a plot

```
e_cos    ← abs(array of iterate(g_cos, 1.0, 30) - L_cos)
e_newton ← abs(array of iterate(g_newton, 1.0, 6) - L_newton)
plot e_cos and e_newton with plt.semilogy (one line each, with a legend)
```


<details><summary><b>Hint</b></summary>

Turn the list into an array first, so you can subtract $L$ from all entries at once: `np.abs(np.array(xs) - L)`.

</details>

In [ ]:
# your code here

**Question:** which curve is a straight line on this plot? Which one bends down? Where does Newton stop improving, and why?


<details><summary><b>Check your answer</b></summary>

The cos iteration gives a straight line: **linear** convergence. Newton bends down steeply: **quadratic** convergence.
Newton hits about $10^{-16}$ after 5 iterations and cannot go lower: that is machine epsilon.

</details>

### Step 3: the factor $\rho$ of linear convergence

For linear convergence $e_{n+1} \approx \rho\, e_n$. Print the ratios $e_{n+1}/e_n$ of the cos iteration:

```
for k from 10 to 15:
    print k and e_cos[k + 1] / e_cos[k]
```

In [ ]:
# your code here

<details><summary><b>Check your answer</b></summary>

The ratios settle near $\rho \approx 0.67$ (theory: $|\sin L| = 0.6736$). Every iteration the error shrinks by a third,
so one extra correct digit takes about $\log(10)/\log(1/0.67) \approx 6$ iterations.

</details>

### Step 4: estimate the order $p$

If $e_{n+1} \approx C\, e_n^{\,p}$, three consecutive errors give an estimate of $p$:

$$p \approx \frac{\log(e_{n+1}/e_n)}{\log(e_n / e_{n-1})}$$

```
function estimate_order(e):
    ps ← empty list
    for k from 1 to (length of e) - 2:
        append log(e[k + 1] / e[k]) / log(e[k] / e[k - 1]) to ps
    return ps
```


<details><summary><b>Hint</b></summary>

"for k from 1 to (length of e) - 2" is `for k in range(1, len(e) - 1):`.

</details>

In [ ]:
def estimate_order(e):
    # your code here
    pass

In [ ]:
# mini-check on made-up errors: 1e-1, 1e-2, 1e-4, 1e-8 is exactly quadratic
p = estimate_order([1e-1, 1e-2, 1e-4, 1e-8])
assert np.allclose(p, [2.0, 2.0])
print("OK")

Now apply it to the real sequences. Use only errors **well above** $10^{-16}$ (near the floor the estimates are noise):

```
print estimate_order of the first 5 errors of Newton   (e_newton[:5])
print estimate_order of e_cos[10:16]
```

In [ ]:
# your code here

<details><summary><b>Check your answer</b></summary>

Newton gives $p \approx 2$ (quadratic), the cos iteration gives $p \approx 1$ (linear).

</details>

### Step 5: when to stop?

In practice $L$ is unknown, so we cannot compute the error. Instead, stop when the **change per iteration** becomes small, and never loop forever:

```
function iterate_until(g, x0, tol, max_iter):
    x ← x0
    repeat for k = 1, 2, ..., max_iter:
        x_new ← g(x)
        if abs(x_new - x) < tol:
            return [x_new, k]              (converged after k iterations)
        x ← x_new
    return [x, max_iter]
```


<details><summary><b>Hint</b></summary>

"repeat for k = 1, 2, ..., max_iter" is `for k in range(1, max_iter + 1):`.

</details>

In [ ]:
def iterate_until(g, x0, tol, max_iter):
    # your code here
    pass

In [ ]:
# mini-check
result_cos = iterate_until(g_cos, 1.0, 1e-10, 1000)
result_newton = iterate_until(g_newton, 1.0, 1e-10, 1000)
print(f"cos iteration: x = {result_cos[0]:.12f} after {result_cos[1]} iterations")
print(f"Newton       : x = {result_newton[0]:.12f} after {result_newton[1]} iterations")
assert abs(result_cos[0] - L_cos) < 1e-9
assert abs(result_newton[0] - L_newton) < 1e-15
assert result_newton[1] < 10 < result_cos[1]
print("Problem 4: passed")

**Question:** how many iterations did each method need for a tolerance of $10^{-10}$? What happens if you ask for `tol = 1e-20`?


<details><summary><b>Check your answer</b></summary>

Newton needs 5 iterations, the cos iteration 58. With `tol = 1e-20` the change per iteration can never get that small (floats near 1 are $2.2 \times 10^{-16}$ apart),
so the loop only ends because of `max_iter`. That is why a maximum number of iterations is essential.

</details>